## Runtime Context (`RunnableConfig`)

In the memory notebook we passed `config = {"configurable": {"thread_id": "1"}}` to the agent.
That `config` dict is LangChain's **runtime context** — a structured object that flows
through every node, tool, and runnable in the chain.

Runtime context solves a real problem: **how do you give a tool information it needs
(user ID, tenant name, feature flags) without making the LLM pass it explicitly?**

**Topics covered:**
* The `RunnableConfig` structure — built-in keys
* `tags`, `metadata`, `run_name` — for tracing and observability
* The `configurable` namespace — your custom key-value store
* `ConfigurableField` — making model parameters runtime-switchable
* Injecting config into `@tool` functions (the killer feature)
* Multi-tenant agent pattern

In [1]:
%run langchain_common.py

mlflow.set_tracking_uri("http://localhost:5000")  # the `mlflow ui` server from the repo setup 
mlflow.set_experiment("langchain_runtime_context_runnableconfig")
mlflow.langchain.autolog() 

/Users/apple/Desktop/Summer'26/Agentic AI /cs4603/wk3_agents/langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.
/Users/apple/Desktop/Summer'26/Agentic AI /cs4603/.venv-cs4603/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MlflowException: API request to endpoint /api/2.0/mlflow/experiments/get-by-name failed with error code 403 != 200. Response body: ''

## 1. The `RunnableConfig` structure

```python
config = {
    # -- Built-in LangChain/LangGraph keys --
    "run_name":        str,        # label shown in MLflow / LangSmith traces
    "tags":            list[str],  # searchable labels attached to a run
    "metadata":        dict,       # arbitrary key-value data stored with the run
    "recursion_limit": int,        # max graph recursion depth (default 25)
    "max_concurrency": int,        # parallel branch limit

    # -- Your custom namespace --
    "configurable": {
        "thread_id": "abc",        # memory: identifies the conversation thread
        # ...any other keys you define
    },
}
```

`configurable` is where you put anything application-specific.
The other keys are consumed by LangChain internals.

## 2. `tags`, `metadata`, `run_name` — observability

These keys annotate a run for filtering and searching in tracing tools like MLflow.
They cost nothing to add and pay off when debugging production issues.

In [14]:
agent = create_agent(llm_noreason)

# Rich config — the agent behaviour is identical,
# but tracing tools can now filter/search by these values.
config = {
    "run_name":  "student-query",
    "tags":      ["cs4603", "wk3", "demo"],
    "metadata":  {"user_id": "student-42", "course": "CS4603", "env": "notebook"},
}

result = agent.invoke(
    {"messages": [HumanMessage(content="What is the capital of Pakistan?")]},
    config,
)
print(result["messages"][-1].content)

The capital of Pakistan is **Islamabad**.

Located in the Potohar Plateau in the north of the country, Islamabad was purpose-built in the 1960s to replace Karachi as the national capital. This move was made to create a more centrally located administrative hub and to alleviate the overcrowding in Karachi, which was the original capital after independence in 1947. Today, Islamabad serves as the seat of the government, housing the Parliament House, the Supreme Court, and the President's residence.


## 3. The `configurable` namespace — your custom context

You already know `thread_id` lives here. Any key-value pair you put in `configurable`
is accessible to every runnable in the chain — including tools.

Common uses:
* `user_id` — identify the current user
* `tenant_id` — identify the organisation in a SaaS app
* `feature_flags` — enable/disable behaviours per user
* `user_tier` — adjust responses for free vs. premium users

In [15]:
# Two different sessions — same agent, different context
session_a = make_thread_config("27000001")
session_b = make_thread_config("27000002")

mem_agent = create_agent(llm_noreason, checkpointer=InMemorySaver())

mem_agent.invoke({"messages": [HumanMessage(content="My favourite city is Lahore.")]}, session_a)
mem_agent.invoke({"messages": [HumanMessage(content="My favourite city is Istanbul.")]}, session_b)

result_a = mem_agent.invoke({"messages": [HumanMessage(content="What is my favourite city?")]}, session_a)
result_b = mem_agent.invoke({"messages": [HumanMessage(content="What is my favourite city?")]}, session_b)

print(f"Ali's session  : {result_a['messages'][-1].content}")
print(f"Sara's session : {result_b['messages'][-1].content}")

Ali's session  : Based on what you told me earlier, your favourite city is **Lahore**.
Sara's session : Based on what you told me earlier, your favorite city is **Istanbul**.


[Trace(trace_id=tr-43e4b9e47d5734435af38ebff3843cca), Trace(trace_id=tr-7844d66fc9f65c6a613e862e5641c9b3), Trace(trace_id=tr-975541943383bec639e2e2aa6edf2185), Trace(trace_id=tr-a77f56483ce0f0953729ed85242eee9e)]

## 4. `ConfigurableField` — runtime-switchable model parameters

`ConfigurableField` lets you declare that a chain parameter (e.g., model temperature)
can be overridden at runtime via `configurable` — without rebuilding the chain.

This is useful when:
* You want a **creative mode** (high temperature) and a **precise mode** (low temperature)
* Different user tiers get different LLM models
* You want to A/B test prompts or models without redeploying

In [16]:
from langchain_core.runnables import ConfigurableField

# Declare temperature as a configurable field with a default
configurable_llm = llm_noreason.configurable_fields(
    temperature=ConfigurableField(
        id="llm_temperature",
        name="LLM Temperature",
        description="0 = deterministic / precise, 1 = creative / random",
    )
)

prompt = "Write a one-sentence tagline for a coffee shop."

# Default temperature (0 — deterministic)
result_precise = configurable_llm.invoke(prompt)
print("Precise (temp=0):  ", result_precise.content)

# High temperature at runtime — no code rebuild needed
result_creative = configurable_llm.invoke(
    prompt,
    config={"configurable": {"llm_temperature": 1.0}},
)
print("Creative (temp=1): ", result_creative.content)

Precise (temp=0):   Start your day with a cup of pure inspiration.
Creative (temp=1):  Brewing moments that turn ordinary days into extraordinary mornings.


Trace(trace_id=tr-f01d9e48c8ece403a224cc90de206911)

## 5. Injecting config into `@tool` functions ⭐

This is the most powerful runtime-context feature.

Add `config: RunnableConfig` as a parameter to any `@tool` function.
The config parameter is injected by LangChain at call time, not by the model. The LLM:

* never sees it in the tool's schema,
* can't set it,
* can't override it,
* doesn't even know it exists.

This lets tools access `user_id`, `tenant_id`, or any other context that was
set when `agent.invoke(...)` was called, without the user mentioning it in their message.

In [17]:
from langchain_core.runnables import RunnableConfig

# Simulated user account database
USER_ACCOUNTS = {
    "user-001": {"name": "Ahmed",  "balance": 5000.00, "tier": "standard"},
    "user-002": {"name": "Sara", "balance": 25000.00, "tier": "premium"},
}

@tool
def get_account_balance(config: RunnableConfig) -> str:
    """
    Get the current account balance for the logged-in user.
    No user ID is required — it is read from the session context automatically.
    """
    # Extract user_id from the injected config — the LLM never supplies this
    user_id = config.get("configurable", {}).get("user_id")

    if not user_id:
        return "Error: no user session found. Please log in."

    account = USER_ACCOUNTS.get(user_id)
    if not account:
        return f"Error: account not found for user_id '{user_id}'."

    return f"Hello {account['name']}! Your balance is ${account['balance']:,.2f} (tier: {account['tier']})."


banking_agent = create_agent(llm_noreason, [get_account_balance])

# Same question — different answer depending on who is logged in
for uid in ["user-001", "user-002", "user-999"]:
    ctx = {"configurable": {"user_id": uid}}
    result = banking_agent.invoke({"messages": [HumanMessage(content="What is my balance?")]}, ctx)
    print(f"[user_id={uid}] {result['messages'][-1].content}")
    print()

[user_id=user-001] Your current balance is $5,000.00.

[user_id=user-002] Your current balance is $25,000.00. You are on the premium tier.

[user_id=user-999] I'm sorry, but I couldn't find an account associated with your user ID. Please check your user ID or contact customer support for assistance.



[Trace(trace_id=tr-7d31c261d7d06174e790298b187c646f), Trace(trace_id=tr-25b5d09ba4e5dad45604d6aef59f1b0a), Trace(trace_id=tr-156c3440a3843e0638ec674c9d0b4272), Trace(trace_id=tr-4b70e6b2b633e19c15a55b993dc0931f)]

## 6. Multi-tenant pattern — full example

Combine config injection, memory, and per-tenant context into a realistic pattern:
one agent instance serves many users, each isolated by `user_id` + `thread_id`.

In [18]:
from pydantic import BaseModel, Field

ORDERS = {
    "user-001": [{"id": "ORD-11", "item": "Laptop",  "status": "Shipped"},
                 {"id": "ORD-12", "item": "Mouse",   "status": "Delivered"}],
    "user-002": [{"id": "ORD-21", "item": "Monitor", "status": "Processing"}],
}

@tool
def list_my_orders(config: RunnableConfig) -> str:
    """List all orders belonging to the currently logged-in user."""
    user_id = config.get("configurable", {}).get("user_id", "")
    orders = ORDERS.get(user_id, [])
    if not orders:
        return "No orders found for your account."
    lines = [f"  {o['id']}: {o['item']} — {o['status']}" for o in orders]
    return "Your orders:\n" + "\n".join(lines)


@tool
def get_account_info(config: RunnableConfig) -> str:
    """Return the profile information for the currently logged-in user."""
    user_id = config.get("configurable", {}).get("user_id", "")
    account = USER_ACCOUNTS.get(user_id)
    if not account:
        return "Account not found."
    return f"Name: {account['name']}, Tier: {account['tier']}, Balance: ${account['balance']:,.2f}"


multi_agent = create_agent(
    llm_noreason,
    [list_my_orders, get_account_info],
    checkpointer=InMemorySaver(),
)

def build_config(user_id: str, session_id: str) -> dict:
    return {"configurable": {"user_id": user_id, "thread_id": session_id}}


# Ali's session
user1_cfg   = build_config("user-001", "user1-session-1")
# Sara's session
user2_cfg  = build_config("user-002", "user2-session-1")

questions = [
    (user1_cfg,  "Show me my orders."),
    (user2_cfg, "What is my account info?"),
    (user1_cfg,  "What is my account balance?"),
]

for cfg, q in questions:
    result = multi_agent.invoke({"messages": [HumanMessage(content=q)]}, cfg)
    uid = cfg["configurable"]["user_id"]
    print(f"[{uid}] Q: {q}")
    print(f"        A: {result['messages'][-1].content}")
    print()

[user-001] Q: Show me my orders.
        A: Here are your orders:

*   **ORD-11**: Laptop — Shipped
*   **ORD-12**: Mouse — Delivered

[user-002] Q: What is my account info?
        A: Your account information is as follows:
- **Name:** Sara
- **Tier:** Premium
- **Balance:** $25,000.00

[user-001] Q: What is my account balance?
        A: Your account balance is $5,000.00.



[Trace(trace_id=tr-711b01d6682bc721a513766c6704c1b9), Trace(trace_id=tr-2c63c24e956b0f426ae8c26702afa2b3), Trace(trace_id=tr-79aae03a3dab2a27ec4e37ca256777e8)]

## 7. `with_config` — setting default config on a runnable

`chain.with_config(config)` returns a new runnable with the given config baked in as defaults.
Callers can still override any key. Useful for creating environment-specific variants
(dev / staging / prod) without passing config manually on every call.

In [19]:
chain = (
    ChatPromptTemplate.from_messages([("human", "{question}")])
    | llm_noreason
    | StrOutputParser()
)

# Bind default metadata for all runs from this variant
prod_chain = chain.with_config(
    run_name="prod-query",
    tags=["production", "cs4603"],
    metadata={"env": "prod"},
)

dev_chain = chain.with_config(
    run_name="dev-query",
    tags=["development", "cs4603"],
    metadata={"env": "dev"},
)

# Both use the same chain logic; tracing tools will distinguish them by tag/metadata
print(prod_chain.invoke({"question": "What is 2+2?"}))
print(dev_chain.invoke({"question": "What is 2+2?"}))

The sum of 2 and 2 is **4**.
The sum of 2 and 2 is **4**.


[Trace(trace_id=tr-98d53b51c710bc86535cd2e323507dc8), Trace(trace_id=tr-8969ad06f13600e46db6a68e5d3d8951)]

## Summary

| Feature | Key API | Use case |
|---|---|---|
| Observability labels | `tags`, `metadata`, `run_name` | Filter traces in MLflow / LangSmith |
| Custom runtime data | `configurable: {...}` | Pass user_id, tenant_id, feature flags |
| Runtime param overrides | `ConfigurableField` | Switch temperature / model per call |
| Config injection in tools | `config: RunnableConfig` param | Multi-tenant data access without LLM involvement |
| Bound defaults | `chain.with_config(...)` | Environment-specific chain variants |

The key insight: **the `config` dict flows invisibly through the entire chain**.
You set it once at the top-level `invoke` call, and every tool and runnable downstream
can read from it — no need to thread extra arguments through every function.